In [1]:
!qiime tools import \
  --type 'SampleData[SequencesWithQuality]' \
  --input-path fastq/manifest/manifest.tsv \
  --output-path demux-single-end.qza \
  --input-format SingleEndFastqManifestPhred33V2

Imported fastq/manifest/manifest.tsv as SingleEndFastqManifestPhred33V2 to demux-single-end.qza


In [2]:
!qiime demux summarize --i-data demux-single-end.qza --o-visualization demux.qzv

Saved Visualization to: demux.qzv


In [3]:
!qiime dada2 denoise-single \
  --i-demultiplexed-seqs ./demux-single-end.qza \
  --p-trunc-len 150 \
  --o-table ./dada2_table.qza \
  --o-representative-sequences ./dada2_rep_set.qza \
  --o-denoising-stats ./dada2_stats.qza

Saved FeatureTable[Frequency] to: ./dada2_table.qza
Saved FeatureData[Sequence] to: ./dada2_rep_set.qza
Saved SampleData[DADA2Stats] to: ./dada2_stats.qza


In [4]:
!qiime metadata tabulate \
  --m-input-file ./dada2_stats.qza  \
  --o-visualization ./dada2_stats.qzv

Saved Visualization to: ./dada2_stats.qzv


In [5]:
!qiime feature-table summarize \
  --i-table ./dada2_table.qza \
  --m-sample-metadata-file sample-metadata.tsv \
  --o-visualization ./dada2_table.qzv


Saved Visualization to: ./dada2_table.qzv


In [6]:
!qiime feature-table tabulate-seqs \
  --i-data dada2_rep_set.qza \
  --o-visualization rep-seqs.qzv

Saved Visualization to: rep-seqs.qzv


In [7]:
!qiime feature-table summarize \
  --i-table dada2_table.qza \
  --m-sample-metadata-file sample-metadata.tsv \
  --o-visualization table.qzv

Saved Visualization to: table.qzv


In [8]:
!qiime phylogeny align-to-tree-mafft-fasttree \
  --i-sequences dada2_rep_set.qza \
  --o-alignment aligned-rep-seqs.qza \
  --o-masked-alignment masked-aligned-rep-seqs.qza \
  --o-tree unrooted-tree.qza \
  --o-rooted-tree rooted-tree.qza

Saved FeatureData[AlignedSequence] to: aligned-rep-seqs.qza
Saved FeatureData[AlignedSequence] to: masked-aligned-rep-seqs.qza
Saved Phylogeny[Unrooted] to: unrooted-tree.qza
Saved Phylogeny[Rooted] to: rooted-tree.qza


In [9]:
!qiime empress tree-plot \
    --i-tree rooted-tree.qza \
    --o-visualization empress.qzv

Saved Visualization to: empress.qzv


In [10]:
!qiime diversity alpha-rarefaction \
  --i-table dada2_table.qza \
  --i-phylogeny rooted-tree.qza \
  --p-max-depth 10000 \
  --m-metadata-file  sample-metadata.tsv \
  --o-visualization alpha-rarefaction.qzv

Saved Visualization to: alpha-rarefaction.qzv


In [11]:
!qiime diversity core-metrics-phylogenetic \
  --i-phylogeny rooted-tree.qza \
  --i-table dada2_table.qza \
  --p-sampling-depth 1200 \
  --m-metadata-file sample-metadata.tsv  \
  --output-dir core-metrics-results-1

Saved FeatureTable[Frequency] to: core-metrics-results-1/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results-1/faith_pd_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results-1/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results-1/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results-1/evenness_vector.qza
Saved DistanceMatrix to: core-metrics-results-1/unweighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results-1/weighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results-1/jaccard_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results-1/bray_curtis_distance_matrix.qza
Saved PCoAResults to: core-metrics-results-1/unweighted_unifrac_pcoa_results.qza
Saved PCoAResults to: core-metrics-results-1/weighted_unifrac_pcoa_results.qza
Saved PCoAResults to: core-metrics-results-1/jaccard_pcoa_results.qza
Saved PCoAResults to: core-metrics-res

In [12]:
!qiime diversity alpha-group-significance \
    --i-alpha-diversity core-metrics-results-1/shannon_vector.qza \
    --m-metadata-file sample-metadata.tsv\
    --o-visualization alpha_groups.qzv

Saved Visualization to: alpha_groups.qzv


In [4]:
!qiime diversity alpha-group-significance \
  --i-alpha-diversity ./core-metrics-results-1/faith_pd_vector.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization ./core-metrics-results-1/faiths_pd_statistics.qzv


Saved Visualization to: ./core-metrics-results-1/faiths_pd_statistics.qzv


In [ ]:
qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results-1/unweighted_unifrac_distance_matrix.qza \
  --m-metadata-file sample-metadata.tsv \
  --m-metadata-column AB_used \
  --o-visualization core-metrics-results-1/unweighted-unifrac-donor-significance.qzv

qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results-1/weighted_unifrac_distance_matrix.qza \
  --m-metadata-file sample-metadata.tsv \
  --m-metadata-column AB_used \
  --o-visualization core-metrics-results-1/weighted-unifrac-donor-significance.qzv

In [8]:
!qiime feature-classifier classify-sklearn \
    --i-reads dada2_rep_set.qza \
    --i-classifier gg-13-8-99-515-806-nb-classifier.qza \
    --o-classification taxa1.qza

Saved FeatureData[Taxonomy] to: taxa1.qza


In [9]:
!qiime taxa barplot \
    --i-table dada2_table.qza \
    --i-taxonomy taxa1.qza \
    --m-metadata-file sample-metadata.tsv \
    --o-visualization taxa_barplot.qzv

Saved Visualization to: taxa_barplot.qzv


In [10]:
!qiime feature-table filter-samples \
  --i-table ./dada2_table.qza \
  --p-min-frequency 1200 \
  --o-filtered-table ./table_2k.qza

Saved FeatureTable[Frequency] to: ./table_2k.qza


In [11]:
!qiime taxa barplot \
  --i-table ./table_2k.qza \
  --i-taxonomy taxa1.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization ./taxa_barplot.qzv

Saved Visualization to: ./taxa_barplot.qzv


In [12]:
!qiime feature-table filter-features \
  --i-table ./table_2k.qza \
  --p-min-frequency 50 \
  --p-min-samples 4 \
  --o-filtered-table ./table_2k_abund.qza

Saved FeatureTable[Frequency] to: ./table_2k_abund.qza


In [13]:
!qiime composition add-pseudocount \
  --i-table ./table_2k_abund.qza \
  --o-composition-table ./table2k_abund_comp.qza


Saved FeatureTable[Composition] to: ./table2k_abund_comp.qza


In [14]:
!qiime composition ancom \
  --i-table ./table2k_abund_comp.qza \
  --m-metadata-file sample-metadata.tsv \
  --m-metadata-column AB_used \
  --o-visualization ./ancom_AB_used.qzv



Saved Visualization to: ./ancom_AB_used.qzv
